# Домашнє завдання до теми 4
**Студентка:** Вох Ірина  
**Дисципліна:** Вступ до машинного навчання (ІПЗ, 5 семестр)

**Мета:** покращити базову модель лінійної регресії на датасеті California Housing з теми 3: очистити дані від викидів за допомогою `zscore`, прибрати одну з сильно корельованих ознак, масштабувати дані, перенавчити модель і порівняти метрики R², MAE, MAPE з базовими.

## 1. Імпорт пакетів

In [1]:
import warnings

import numpy as np
import pandas as pd
from scipy.stats import zscore
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)

## 2. Завантаження датасету

In [2]:
california_housing = fetch_california_housing(as_frame=True)
data = california_housing["frame"]

print(data.shape)
data.head()

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.325,41.0,6.984,1.024,322.0,2.556,37.88,-122.23,4.526
1,8.301,21.0,6.238,0.972,2401.0,2.110,37.86,-122.22,3.585
2,7.257,52.0,8.288,1.073,496.0,2.802,37.85,-122.24,3.521
3,5.643,52.0,5.817,1.073,558.0,2.548,37.85,-122.25,3.413
4,3.846,52.0,6.282,1.081,565.0,2.181,37.85,-122.25,3.422


Первинний EDA виконано в темі 3, тому тут не повторюємо його.

### Базова модель (як у темі 3)
Щоб коректно порівняти результати, відтворюю базову модель з теми 3 на тих самих даних і з тим самим розбиттям: без очищення, усі 8 ознак, стандартизація, лінійна регресія.

In [3]:
TARGET = "MedHouseVal"
RANDOM_STATE = 42


def train_and_evaluate(df):
    X = df.drop(columns=TARGET)
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)

    model = LinearRegression().fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r_sq = model.score(X_train, y_train)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    return r_sq, mae, mape


r_sq, mae, mape = train_and_evaluate(data)
print(f"R2: {r_sq:.2f} | MAE: {mae:.2f} | MAPE: {mape:.2f}")

R2: 0.61 | MAE: 0.53 | MAPE: 0.32


## 3. Додаткова обробка даних

### 3.1. Очищення від викидів
З розподілів ознак у темі 3 видно довгі «хвости» у колонках `AveRooms`, `AveBedrms`, `AveOccup` і `Population`. Наприклад, `AveOccup` сягає понад 1200 мешканців на домогосподарство, що фізично неможливо для звичайних районів.

Для пошуку аномалій використовую `zscore()` з `scipy`. Він показує, на скільки стандартних відхилень значення віддалене від середнього. Значення з `|z| > 3` вважаю викидом.

In [4]:
features_of_interest = ["AveRooms", "AveBedrms", "AveOccup", "Population"]

z_scores = data[features_of_interest].apply(zscore)
outliers = z_scores.abs() > 3

print("Кількість викидів у кожній колонці (перший прохід):")
print(outliers.sum())
print(f"Рядків хоча б з одним викидом: {outliers.any(axis=1).sum()}")

Кількість викидів у кожній колонці (перший прохід):
AveRooms      133
AveBedrms     145
AveOccup        8
Population    342
dtype: int64
Рядків хоча б з одним викидом: 505


**Правило видалення:** рядок видаляється, якщо хоча б в одній із чотирьох колонок значення є викидом (`|z| > 3`, перевірка через `any()`).

Після видалення найбільших викидів середнє і стандартне відхилення змінюються, і частина значень, які раніше «ховались» за екстремальними, теж стає аномальною. Тому застосовую правило **повторно, доки викидів не залишиться**.

In [5]:
data_clean = data.copy()
n_pass = 0

while True:
    z = data_clean[features_of_interest].apply(zscore)
    mask = (z.abs() > 3).any(axis=1)
    if not mask.any():
        break
    n_pass += 1
    print(f"Прохід {n_pass}: видалено {mask.sum()} рядків")
    data_clean = data_clean[~mask]

removed = len(data) - len(data_clean)
print(f"\nВсього видалено {removed} рядків ({removed / len(data):.1%}), залишилось {len(data_clean)}")

Прохід 1: видалено 505 рядків
Прохід 2: видалено 960 рядків
Прохід 3: видалено 805 рядків
Прохід 4: видалено 366 рядків
Прохід 5: видалено 167 рядків
Прохід 6: видалено 86 рядків
Прохід 7: видалено 32 рядків
Прохід 8: видалено 15 рядків
Прохід 9: видалено 7 рядків
Прохід 10: видалено 1 рядків

Всього видалено 2944 рядків (14.3%), залишилось 17696


In [6]:
data_clean[features_of_interest].describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
AveRooms,5.227,1.126,1.875,8.602
AveBedrms,1.047,0.069,0.842,1.254
AveOccup,2.866,0.653,0.971,4.821
Population,1260.345,630.619,5.000,3152.000


### 3.2. Видалення сильно корельованої ознаки
Матриця кореляції ознак з теми 3 (на вихідних даних):

In [7]:
corr = data.corr()
corr.round(2)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
MedInc,1.00,-0.12,0.33,-0.06,0.00,0.02,-0.08,-0.02,0.69
HouseAge,-0.12,1.00,-0.15,-0.08,-0.30,0.01,0.01,-0.11,0.11
AveRooms,0.33,-0.15,1.00,0.85,-0.07,-0.00,0.11,-0.03,0.15
AveBedrms,-0.06,-0.08,0.85,1.00,-0.07,-0.01,0.07,0.01,-0.04
Population,0.00,-0.30,-0.07,-0.07,1.00,0.07,-0.11,0.10,-0.02
AveOccup,0.02,0.01,-0.00,-0.01,0.07,1.00,0.00,0.00,-0.02
Latitude,-0.08,0.01,0.11,0.07,-0.11,0.00,1.00,-0.92,-0.14
Longitude,-0.02,-0.11,-0.03,0.01,0.10,0.00,-0.92,1.00,-0.05
MedHouseVal,0.69,0.11,0.15,-0.04,-0.02,-0.02,-0.14,-0.05,1.00


Найсильніше між собою корелюють пари:
- `Latitude` і `Longitude` (≈ -0.9). Але це координати, разом вони задають місце розташування, яке сильно впливає на ціну, тому прибирати одну з них не варто.
- `AveRooms` і `AveBedrms` (≈ 0.85): середня кількість кімнат і спалень у будинку майже дублюють одна одну (мультиколінеарність).

З пари `AveRooms` / `AveBedrms` видаляю **`AveBedrms`**, бо вона слабше пов'язана з цільовою змінною `MedHouseVal`, ніж `AveRooms`:

In [8]:
print(corr.loc[["AveRooms", "AveBedrms"], ["AveRooms", "AveBedrms", TARGET]].round(2))

print("\nКореляція AveRooms / AveBedrms після очищення:",
      round(data_clean["AveRooms"].corr(data_clean["AveBedrms"]), 2))

data_final = data_clean.drop(columns="AveBedrms")
print("\nОзнаки для моделі:", list(data_final.drop(columns=TARGET).columns))

           AveRooms  AveBedrms  MedHouseVal
AveRooms       1.00       0.85         0.15
AveBedrms      0.85       1.00        -0.04

Кореляція AveRooms / AveBedrms після очищення: -0.01

Ознаки для моделі: ['MedInc', 'HouseAge', 'AveRooms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


## 4–6. Розбиття на train/test, масштабування і побудова моделі

In [9]:
X = data_final.drop(columns=TARGET)
y = data_final[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

pd.Series(model.coef_, index=X.columns).sort_values()

Latitude     -0.942
Longitude    -0.865
AveOccup     -0.295
AveRooms     -0.076
Population    0.063
HouseAge      0.147
MedInc        0.763
dtype: float64

## 7. Оцінка моделі

In [10]:
r_sq_upd = model.score(X_train_scaled, y_train)
mae_upd = mean_absolute_error(y_test, y_pred)
mape_upd = mean_absolute_percentage_error(y_test, y_pred)

print(f"R2: {r_sq_upd:.2f} | MAE: {mae_upd:.2f} | MAPE: {mape_upd:.2f}")

R2: 0.67 | MAE: 0.48 | MAPE: 0.28


## 8. Порівняння з базовою моделлю

In [11]:
comparison = pd.DataFrame(
    {
        "Базова модель (тема 3)": [r_sq, mae, mape],
        "Покращена модель": [r_sq_upd, mae_upd, mape_upd],
    },
    index=["R2", "MAE", "MAPE"],
)
comparison["Зміна"] = comparison["Покращена модель"] - comparison["Базова модель (тема 3)"]
comparison.round(3)

,Базова модель (тема 3),Покращена модель,Зміна
R2,0.613,0.671,0.058
MAE,0.535,0.479,-0.056
MAPE,0.320,0.280,-0.040


In [12]:
print(f"MAPE покращилась на {(mape - mape_upd) * 100:.1f} відсоткових пункти")

MAPE покращилась на 4.0 відсоткових пункти


## Висновки

1. **Очищення від викидів** дало основний приріст якості. Екстремальні значення `AveRooms`, `AveBedrms`, `AveOccup`, `Population` (зокрема, неможливі значення на кшталт сотень мешканців на домогосподарство) сильно «тягнули» лінійну модель, бо метод найменших квадратів дуже чутливий до викидів. Видалення через `zscore` з повторними проходами прибрало близько 14% рядків, але це були переважно нетипові райони.
2. **Видалення `AveBedrms`** прибрало мультиколінеарність з `AveRooms` (кореляція 0.85 на вихідних даних). Цікаво, що після очищення ця кореляція майже зникла: її значною мірою створювали саме викиди (райони з аномально великими будинками мають і багато кімнат, і багато спалень). На точність прогнозу видалення ознаки майже не впливає, але робить модель простішою, а коефіцієнти стабільнішими.
3. **Результат:** R² зріс (модель пояснює більшу частку дисперсії ціни), MAE зменшилась, а MAPE покращилась приблизно на 4 відсоткові пункти, що відповідає очікуваному діапазону 3–5 п.п.
4. **Обмеження:** очищення зменшило вибірку, тому модель гірше працюватиме на «екстремальних» районах. Також залишається проблема обрізаної цільової змінної (`MedHouseVal` обмежена значенням 5) і нелінійних залежностей (наприклад, від координат), які лінійна регресія не вловлює. Для подальшого покращення варто спробувати нелінійні моделі або генерацію нових ознак.